# Python & Pandas Debugging — Data Wrangling

This guide is structured for a **hands-on 60-minute technical interview** focused on data wrangling with Pandas. You'll encounter real bugs, silent data errors, and merge mishaps. This notebook teaches you how to read tracebacks, classify errors, apply fix patterns, and review code like a professional analyst.

<hr style="border: 3px solid black;">

## Table of Contents

1. [How to Read a Python Traceback](#section-1)
2. [Error Classification Decision Tree](#section-2)
3. [Pattern Library: 8 Bug Types](#section-3)
   - [Pattern A: TypeError](#pattern-a)
   - [Pattern B: KeyError & IndexError](#pattern-b)
   - [Pattern C: ValueError](#pattern-c)
   - [Pattern D: SettingWithCopyWarning](#pattern-d)
   - [Pattern E: Merge/Join Issues (Silent Errors)](#pattern-e)
   - [Pattern F: Shape & Dimension Errors](#pattern-f)
   - [Pattern G: Silent Data Errors](#pattern-g)
   - [Pattern H: Import & Environment Errors](#pattern-h)
4. [Code Review Checklist](#section-4)
5. [Debugging Workflow Under Pressure](#section-5)
6. [Common Interview Traps](#section-6)

<hr style="border: 3px solid black;">

# Section 1: How to Read a Python Traceback {#section-1}

## The Golden Rule: Read Bottom-Up

Python tracebacks look scary. They're not. They tell a linear story of where your code broke. **Always read the last 3 lines first.**

---

## The 3-Second Read

1. **Error Type** (last line): What went wrong? `TypeError`, `KeyError`, `ValueError`?
2. **Error Message** (last line): Why did it go wrong? This is gold.
3. **The Code Line** (in the traceback): Which line of *your* code caused it?

---

## Real Example Traceback

```
Traceback (most recent call last):
  File "analysis.py", line 42, in <module>
    result = df['name'] + 5
             ~~~~~~~~~~~~^~~
TypeError: unsupported operand type(s) for +: 'str' and 'int'
```

**Labeled breakdown:**
- **`Traceback (most recent call last):`** — Start here, then read DOWN.
- **`File "analysis.py", line 42`** — Your code, line 42.
- **`result = df['name'] + 5`** — The exact code that broke.
- **`TypeError:`** — Error type (not a KeyError, not a ValueError).
- **`unsupported operand type(s) for +: 'str' and 'int'`** — The why: you tried to add a string to an int.

---

## Anatomy Table

| Part | What It Tells You | Where to Look |
|------|-------------------|----------------|
| Error Type (`TypeError`, `KeyError`, etc.) | What category of mistake | Last line |
| Error Message | Specific reason it failed | Last line (after the colon) |
| File path & line number | Exactly where in your code | Middle of traceback |
| Code context (the indented line) | The exact line that broke | Middle of traceback |
| Stack trace (the flow above) | How Python got to that line | First few lines (optional to read) |

---

## Interview Tip: Read Errors Out Loud

In a real interview, **say the error message out loud** after you see it. Don't stare silently.

❌ Bad: Stare at the screen for 10 seconds.

✅ Good: "I see a TypeError on line 42. It says I'm trying to add a string to an integer. That's because the 'name' column is text, not numeric."

This shows confidence and gives the interviewer insight into your thinking.

<hr style="border: 3px solid black;">

# Section 2: Error Classification Decision Tree {#section-2}

When you hit an error, use this tree to navigate to the right fix pattern.

<div class="fc">
  <div class="fc-node fc-start">Error Occurred — Read the Error Type<br/>(last line of traceback)</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">TypeError?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern A</strong> (A1–A4)<br/>"str" + int? Wrong arg type? df.columns()? Unsupported ops?</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">KeyError or IndexError?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern B</strong> (B1–B4)<br/>Column typo? Wrong label in .loc? MultiIndex issue?</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">ValueError?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern C</strong> (C1–C4)<br/>Length mismatch? String-to-float? Merge overlap? Pivot duplicates?</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">SettingWithCopyWarning?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern D</strong> (D1–D3)<br/>Chained indexing? Missing .copy()? Use .loc[] for assignment.</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">ImportError or ModuleNotFoundError?</div>
  <div class="fc-node fc-good">YES → <strong>Pattern H</strong><br/>Package not installed? Wrong function name? Version conflict?</div>
</div>

---

## Silent Errors (No Traceback!)

These are tricky. Code runs, no error, but your data is wrong.

<div class="fc">
  <div class="fc-node fc-start">Code runs but results look wrong?</div>
  <div class="fc-arrow">▼</div>

  <div class="fc-node fc-start">Merge produced more rows than expected?</div>
  <div class="fc-node fc-warn">CHECK → <strong>Pattern E</strong> — Fan-out, NaN, dtype mismatch in merge keys</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Result is empty or all NaN?</div>
  <div class="fc-node fc-warn">CHECK → <strong>Pattern E or G</strong> — Merge key mismatch, dtypes, NaN handling</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Shape/dimension mismatch?</div>
  <div class="fc-node fc-warn">CHECK → <strong>Pattern F</strong> — DataFrame vs Series, axis confusion, boolean indexing</div>
  <div class="fc-arrow fc-else">if no ▼</div>

  <div class="fc-node fc-start">Values look off? (wrong dtype, NaN propagation)</div>
  <div class="fc-node fc-warn">CHECK → <strong>Pattern G</strong> — Silent coercion, object columns, .str accessor</div>
</div>

---

## Key Insight: Hard Errors vs. Silent Errors

**Hard Errors** (you see a traceback):
- Easy to catch during testing
- Stop execution
- Examples: `TypeError`, `KeyError`, `ValueError`

**Silent Errors** (code runs, result is wrong):
- **Harder to catch**—no traceback to alert you
- Must inspect data with `.shape`, `.head()`, `.dtypes`
- Examples: merge fan-out, dtype coercion, NaN propagation
- **Interview focus**: Interviewers love testing if you check dimensions and dtypes *before* declaring code correct.

<hr style="border: 3px solid black;">

# Section 3: Pattern Library — 8 Bug Types {#section-3}

Below are the 8 most common Pandas debugging patterns you'll encounter in a technical interview. Each pattern has 3-5 subpatterns with:
- The error message you'll see (or not see)
- Code that breaks
- Explanation of what went wrong
- The fix
- Verification

<hr style="border: 2px solid black;">

# Pattern A: TypeError {#pattern-a}

"You tried to do something with incompatible types."

---

## A1: String + Integer (Column Concatenation)

**Error:**
```
TypeError: unsupported operand type(s) for +: 'str' and 'int'
```

**Broken code:**
```python
df = pd.DataFrame({'name': ['Alice', 'Bob'], 'age': [30, 25]})
result = df['name'] + 5  # Trying to add int to string column
```

**Explanation:**
You're treating a string column like a numeric column. Pandas preserves dtypes strictly.

**Fix:**
```python
# If you meant to concatenate: convert to string
result = df['name'] + df['age'].astype(str)

# If you meant to add: convert name to numeric (if it represents numbers)
result = pd.to_numeric(df['name'], errors='coerce') + 5
```

**Verify:**
```python
print(result.dtype)  # Should be 'object' (string) or 'float64' (numeric)
print(result.head())
```

---

## A2: Passing Wrong Argument Type to a Function

**Error:**
```
TypeError: merge() got an unexpected keyword argument 'on='
```
(or similar)

**Broken code:**
```python
df_left.merge(df_right, on=['id'], how='outer')  # OK
df_left.merge(df_right, on='id', left_on=['id2'])  # Error: conflicting args
```

**Explanation:**
The function signature expects certain arg types or combinations. You passed something it doesn't accept.

**Fix:**
```python
# Read the function signature
help(pd.merge)

# Correct usage:
df_left.merge(df_right, left_on='id2', right_on='id', how='outer')
```

---

## A3: Calling a Non-Callable (Missing Parentheses)

**Error:**
```
TypeError: 'Index' object is not callable
```

**Broken code:**
```python
col_names = df.columns()  # columns is a property, not a method
shape_tuple = df.shape()  # shape is a tuple property, not a method
```

**Explanation:**
`.columns` and `.shape` are properties (attributes), not methods. Don't use `()` on them.

**Fix:**
```python
col_names = df.columns  # No parentheses
shape_tuple = df.shape  # No parentheses
```

---

## A4: Unsupported Operand Types in Pandas Operations

**Error:**
```
TypeError: unsupported operand type(s) for /: 'str' and 'int'
```

**Broken code:**
```python
df['revenue'] = df['sales'] / df['units']  # But 'sales' is object dtype (strings!)
df['ratio'] = df['date1'] - df['date2']  # Date columns not parsed as datetime
```

**Explanation:**
You're doing arithmetic on columns that aren't numeric, or date math on string columns.

**Fix:**
```python
# Convert to numeric first
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')
df['revenue'] = df['sales'] / df['units']

# Convert to datetime first
df['date1'] = pd.to_datetime(df['date1'])
df['date2'] = pd.to_datetime(df['date2'])
df['ratio'] = df['date1'] - df['date2']
```

<hr style="border: 2px solid black;">

# Pattern B: KeyError & IndexError {#pattern-b}

"You tried to access a column or row that doesn't exist (or exists under a different name)."

---

## B1: Column Name Typo

**Error:**
```
KeyError: 'naem'
```

**Broken code:**
```python
df = pd.DataFrame({'name': ['Alice', 'Bob'], 'age': [30, 25]})
print(df['naem'])  # Typo: 'naem' instead of 'name'
```

**Explanation:**
The column you're trying to access doesn't exist. Pandas is case-sensitive and doesn't tolerate typos.

**Fix:**
```python
# Always check column names first
print(df.columns)
# Output: Index(['name', 'age'], dtype='object')

# Use the correct name
print(df['name'])

# Or use tab-completion in Jupyter (when using df.column_name syntax)
df.name  # Works if 'name' is a valid Python identifier
```

---

## B2: Accessing Column After Rename or Merge Changed Names

**Error:**
```
KeyError: 'user_id'
```

**Broken code:**
```python
df = df.rename(columns={'user_id': 'customer_id'})
print(df['user_id'])  # This column no longer exists!

# Or after a merge:
df = df_orders.merge(df_users, on='id')
print(df['user_id'])  # May not exist; might have been 'user_id_x' or 'user_id_y'
```

**Explanation:**
After rename or merge, column names change. You're using the old name.

**Fix:**
```python
# Always check columns after transformations
df = df.rename(columns={'user_id': 'customer_id'})
print(df.columns)  # Now 'customer_id' exists, 'user_id' doesn't
print(df['customer_id'])  # Use the new name

# After merge, check for _x and _y suffixes
df = df_orders.merge(df_users, on='id')
print(df.columns)  # Check what's there
print(df['user_id_x'])  # Or df['user_id_y']
```

---

## B3: Using .loc with Labels vs .iloc with Integer Positions

**Error:**
```
KeyError: 0  (or similar)
```

**Broken code:**
```python
df = pd.DataFrame({'a': [10, 20, 30]}, index=['x', 'y', 'z'])
print(df.loc[0])  # Error! 0 is not in the index
print(df.iloc[0])  # OK: first row

print(df.loc['x'])  # OK: row with label 'x'
print(df.iloc['x'])  # Error! String not a position
```

**Explanation:**
- `.loc[]` uses **label-based indexing** (index values, column names)
- `.iloc[]` uses **integer position indexing** (0, 1, 2, ...)

**Fix:**
```python
df = pd.DataFrame({'a': [10, 20, 30]}, index=['x', 'y', 'z'])
print(df.loc['x'])   # Row labeled 'x'
print(df.iloc[0])    # First row (position 0)
print(df.loc[df['a'] > 15])  # Boolean indexing with .loc
```

---

## B4: MultiIndex Column Access After groupby

**Error:**
```
KeyError: 'sales'
```

**Broken code:**
```python
grouped = df.groupby('category').agg({'sales': 'sum', 'units': 'mean'})
print(grouped['sales'])  # Works—single-level column

# But if you use a list of agg functions:
grouped = df.groupby('category').agg({'sales': ['sum', 'mean']})
print(grouped['sales'])  # Now 'sales' is a MultiIndex column! This works but...
print(grouped['sales']['sum'])  # Access the 'sum' level
```

**Explanation:**
When you aggregate with multiple functions per column, Pandas creates a MultiIndex column structure.

**Fix:**
```python
# Check the column structure
print(grouped.columns)
# Output: MultiIndex([('sales', 'sum'), ('sales', 'mean'), ...], ...)

# Access via tuple or chaining
print(grouped[('sales', 'sum')])  # Tuple access
print(grouped['sales']['sum'])    # Chained access

# Or flatten columns after aggregation
grouped.columns = ['_'.join(col).strip() for col in grouped.columns.values]
print(grouped['sales_sum'])
```

<hr style="border: 2px solid black;">

# Pattern C: ValueError {#pattern-c}

"The value you provided doesn't match the operation's requirements."

---

## C1: Length Mismatch When Assigning New Column

**Error:**
```
ValueError: Length of values (5) does not match length of index (10)
```

**Broken code:**
```python
df = pd.DataFrame({'id': [1, 2, 3, 4, 5], 'name': ['A', 'B', 'C', 'D', 'E']})
new_values = [10, 20, 30]  # Only 3 values
df['amount'] = new_values  # Error! df has 5 rows
```

**Explanation:**
When assigning a new column, its length must match the DataFrame's row count.

**Fix:**
```python
# Make sure lists/Series have the same length as df
new_values = [10, 20, 30, 40, 50]  # 5 values for 5 rows
df['amount'] = new_values

# Or use a Series with matching index
df['amount'] = pd.Series([10, 20, 30, 40, 50], index=df.index)

# Verify
print(len(df) == len(new_values))  # Should be True
```

---

## C2: Cannot Convert String to Float (Dirty Data)

**Error:**
```
ValueError: could not convert string to float: '$1,234.56'
```

**Broken code:**
```python
df['price'] = pd.to_numeric(df['price_str'])  # But price_str has '$' and commas
```

**Explanation:**
The string contains special characters that prevent numeric conversion.

**Fix:**
```python
# Remove non-numeric characters first
df['price'] = df['price_str'].str.replace('$', '').str.replace(',', '')
df['price'] = pd.to_numeric(df['price'])

# Or use errors='coerce' to convert bad values to NaN
df['price'] = pd.to_numeric(df['price_str'], errors='coerce')

# Check for NaN (which came from unconvertible strings)
print(df['price'].isna().sum())  # How many couldn't be converted?
```

---

## C3: merge() — Columns Overlap with No Suffix

**Error:**
```
ValueError: suffixes not specified and overlapping columns exist
```

**Broken code:**
```python
df_left = pd.DataFrame({'id': [1, 2], 'amount': [100, 200]})
df_right = pd.DataFrame({'id': [1, 2], 'amount': [50, 75]})
result = df_left.merge(df_right, on='id')  # Error! Both have 'amount' column
```

**Explanation:**
The two DataFrames have a common column name (other than the join key), and Pandas doesn't know how to name the resulting columns.

**Fix:**
```python
# Option 1: Add suffixes
result = df_left.merge(df_right, on='id', suffixes=('_left', '_right'))

# Option 2: Rename before merging
df_right = df_right.rename(columns={'amount': 'amount_right'})
result = df_left.merge(df_right, on='id')

# Verify
print(result.columns)
```

---

## C4: Reshape/Pivot Errors (Duplicate Index Entries)

**Error:**
```
ValueError: Index contains duplicate entries, cannot reshape
```

**Broken code:**
```python
df = pd.DataFrame({
    'date': ['2025-01-01', '2025-01-01', '2025-01-02'],
    'product': ['A', 'A', 'A'],
    'sales': [100, 50, 200]  # Two rows with same (date, product)
})
pivot = df.pivot(index='date', columns='product', values='sales')  # Error!
```

**Explanation:**
Pivot requires unique (index, column) combinations. You have duplicates.

**Fix:**
```python
# Option 1: Aggregate duplicates first
df = df.groupby(['date', 'product'])['sales'].sum().reset_index()
pivot = df.pivot(index='date', columns='product', values='sales')

# Option 2: Use pivot_table (handles duplicates automatically)
pivot = df.pivot_table(index='date', columns='product', values='sales', aggfunc='sum')

# Verify
print(pivot)
```

<hr style="border: 2px solid black;">

# Pattern D: SettingWithCopyWarning {#pattern-d}

"You're modifying a view of a DataFrame instead of a copy, and Pandas isn't sure what you're doing."

---

## D1: Chained Indexing (The Classic Trap)

**Warning:**
```
SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame.
```

**Broken code:**
```python
df = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'salary': [50000, 60000, 55000]
})
df[df['salary'] > 55000]['salary'] = df[df['salary'] > 55000]['salary'] + 1000  # WARNING!
```

**Explanation:**
You're doing `df[df['salary'] > 55000]` (creates a filtered view), then `['salary']` (selects column), then `= value` (assigns). Pandas can't tell if you're modifying the original DataFrame or just a temporary view.

**Why it matters:**
- The assignment might not persist
- Future Pandas versions might change behavior
- You're modifying ambiguous data

**Fix:**
```python
# Use .loc[] for unambiguous assignment
df.loc[df['salary'] > 55000, 'salary'] = df.loc[df['salary'] > 55000, 'salary'] + 1000

# Or
df.loc[df['salary'] > 55000, 'salary'] += 1000

# Verify
print(df)
```

---

## D2: Modifying a Slice Without .copy()

**Warning:**
```
SettingWithCopyWarning: ...
```

**Broken code:**
```python
df = pd.DataFrame({'a': [1, 2, 3, 4, 5], 'b': [10, 20, 30, 40, 50]})
subset = df[df['a'] > 2]  # Creates a view
subset['b'] = subset['b'] * 2  # Trying to modify the view
```

**Explanation:**
`df[df['a'] > 2]` returns a view, not a copy. Modifying it triggers the warning.

**Fix:**
```python
# Option 1: Explicitly copy
subset = df[df['a'] > 2].copy()
subset['b'] = subset['b'] * 2  # Now safe

# Option 2: Use .loc[] to modify in place
df.loc[df['a'] > 2, 'b'] = df.loc[df['a'] > 2, 'b'] * 2

# Verify
print(df)
```

---

## D3: The Correct Way — .loc[] for Assignment

**Best practice:**
```python
df = pd.DataFrame({
    'department': ['Sales', 'IT', 'Sales', 'HR'],
    'salary': [50000, 70000, 55000, 48000]
})

# Raise all Sales salaries by 10%
df.loc[df['department'] == 'Sales', 'salary'] *= 1.10

# Set bonus for high earners
df.loc[df['salary'] > 60000, 'bonus'] = 5000

# Verify
print(df)
```

**Pattern to remember:**
```python
df.loc[row_condition, column_name] = new_value
```
- `row_condition`: Boolean Series (e.g., `df['age'] > 30`)
- `column_name`: String or list of strings
- `new_value`: Scalar, Series, or DataFrame

---

## D4: Why This Matters (Views vs Copies)

**View:** Points to the original data. Changes might propagate.
```python
subset = df[df['a'] > 2]  # View of df
```

**Copy:** Independent data. Changes don't affect the original.
```python
subset = df[df['a'] > 2].copy()  # Independent copy
```

**Interview tip:** Explicitly use `.copy()` or `.loc[]` to show you understand the difference.

<hr style="border: 2px solid black;">

# Pattern E: Merge/Join Issues (Silent Errors) {#pattern-e}

"The merge ran, produced output, but the result is wrong. No error message to guide you."

**This is the pattern interviewers care most about. Merges are where silent data corruption happens.**

---

## E1: Fan-Out — Merge Produces More Rows Than Expected

**Silent error: Row count increased unexpectedly.**

**Example:**
```python
df_orders = pd.DataFrame({
    'customer_id': [1, 1, 2, 3, 3],  # Customer 1 has 2 orders, customer 3 has 2 orders
    'order_amount': [100, 150, 200, 50, 75]
})

df_customers = pd.DataFrame({
    'customer_id': [1, 2, 3],
    'customer_name': ['Alice', 'Bob', 'Charlie']
})

# This merge is 1:many (one customer, many orders)
result = df_orders.merge(df_customers, on='customer_id')
print(len(result))  # Expected 5, got 5 ✓ (OK in this case)

# But what if the key is non-unique on BOTH sides?
df_products = pd.DataFrame({
    'product_id': [1, 1, 2],  # Product 1 appears twice!
    'category': ['Electronics', 'Electronics', 'Food']
})

df_purchases = pd.DataFrame({
    'product_id': [1, 1, 2],  # Product 1 appears twice here too!
    'qty': [5, 3, 10]
})

result = df_products.merge(df_purchases, on='product_id')
print(len(result))  # Expected 3, got 5! (2*2 + 1 = 5)
# Product 1 from df_products matches with both product 1 rows in df_purchases
```

**Explanation:**
When the join key is duplicated on both sides, Cartesian product happens. Each row from one side matches all rows with that key on the other side.

**Fix:**
```python
# 1. Check key uniqueness BEFORE merging
print("Left key uniqueness:")
print(df_products['product_id'].duplicated().sum())  # Should be 0

print("Right key uniqueness:")
print(df_purchases['product_id'].duplicated().sum())  # Should be 0

# 2. If duplicates exist, aggregate first
df_products_clean = df_products.groupby('product_id').first().reset_index()
df_purchases_clean = df_purchases.groupby('product_id').sum().reset_index()
result = df_products_clean.merge(df_purchases_clean, on='product_id')

# 3. Always verify row count
assert len(result) <= max(len(df_products), len(df_purchases)), "Row explosion!"
```

---

## E2: Unexpected NaN After Merge (Key Mismatch, Dtype Mismatch)

**Silent error: Merge succeeds, but many rows become NaN.**

**Example 1: Dtype mismatch (int vs float)**
```python
df_left = pd.DataFrame({
    'id': [1, 2, 3],  # int64
    'value_left': [10, 20, 30]
})

df_right = pd.DataFrame({
    'id': [1.0, 2.0, 3.0],  # float64 (came from a CSV read incorrectly)
    'value_right': [100, 200, 300]
})

result = df_left.merge(df_right, on='id')
print(result)  # No rows! Keys didn't match (int != float)
```

**Example 2: Actual value mismatch**
```python
df_left = pd.DataFrame({
    'user_id': [1, 2, 3, 4],
    'amount': [100, 200, 300, 400]
})

df_right = pd.DataFrame({
    'user_id': [1, 2, 5, 6],  # Users 3, 4 not in right table
    'name': ['Alice', 'Bob', 'Eve', 'Frank']
})

result = df_left.merge(df_right, on='user_id', how='left')
print(result)
# Rows for users 3, 4 will have NaN in 'name'
```

**Fix:**
```python
# 1. Check dtypes before merge
print(df_left.dtypes)
print(df_right.dtypes)
# Make sure join keys have the same dtype
df_right['id'] = df_right['id'].astype('int64')

# 2. Check for NaN in join keys
print(df_left['id'].isna().sum())
print(df_right['id'].isna().sum())
# NaN values don't match anything, even other NaNs

# 3. Do a test merge to count matches
result = df_left.merge(df_right, on='id')
print(f"Left: {len(df_left)}, Right: {len(df_right)}, Result: {len(result)}")
# If result is much smaller, there's a mismatch

# 4. Diagnose which keys didn't match
unmatched_left = df_left[~df_left['id'].isin(df_right['id'])]
print(f"Unmatched in left: {len(unmatched_left)} rows")
print(unmatched_left)
```

---

## E3: Duplicate Column Names After Merge (_x, _y Suffixes)

**Silent error: Merge succeeds, but column names are cryptic.**

**Example:**
```python
df_left = pd.DataFrame({
    'id': [1, 2, 3],
    'amount': [100, 200, 300],  # This column exists in both
    'date': ['2025-01-01', '2025-01-02', '2025-01-03']
})

df_right = pd.DataFrame({
    'id': [1, 2, 3],
    'amount': [1000, 2000, 3000],  # Different values, same name
    'currency': ['USD', 'USD', 'EUR']
})

result = df_left.merge(df_right, on='id', suffixes=('_left', '_right'))
print(result.columns)
# Output: ['id', 'amount_left', 'date', 'amount_right', 'currency']
```

**Fix:**
```python
# 1. Rename columns before merge for clarity
df_left = df_left.rename(columns={'amount': 'amount_orders'})
df_right = df_right.rename(columns={'amount': 'amount_inventory'})
result = df_left.merge(df_right, on='id')  # No suffixes needed

# 2. Or use explicit suffixes
result = df_left.merge(df_right, on='id', suffixes=('_orders', '_inventory'))

# 3. Always check result columns
print(result.columns)
```

---

## E4: Left Merge Dropping Rows When Key Has NaN

**Silent error: left merge should keep all left rows, but doesn't.**

**Example:**
```python
df_left = pd.DataFrame({
    'id': [1, 2, None, 4],  # Row 3 has NaN id
    'value': [10, 20, 30, 40]
})

df_right = pd.DataFrame({
    'id': [1, 2, 4],
    'name': ['A', 'B', 'D']
})

result = df_left.merge(df_right, on='id', how='left')
print(len(result))  # Expected 4, got 3! Row with NaN id is gone!
print(result)
```

**Explanation:**
Pandas merge treats NaN as "no match," even in left merges. The row with NaN disappears.

**Fix:**
```python
# 1. Check for NaN in join keys BEFORE merge
print(df_left['id'].isna().sum())  # Should be 0
print(df_right['id'].isna().sum())  # Should be 0

# 2. Fill or handle NaN in keys
df_left['id'] = df_left['id'].fillna(-1)  # Use sentinel value
result = df_left.merge(df_right, on='id', how='left')

# 3. Or explicitly combine with unmatched rows
merged = df_left.merge(df_right, on='id', how='left')
unmatched = df_left[df_left['id'].isna()]
result = pd.concat([merged, unmatched])  # Ensure no rows are lost

# Verify
assert len(result) == len(df_left), "Lost rows in merge!"
```

---

## Diagnostic Checklist for Merge Issues

Always run this after a merge:
```python
# Before
print(f"Left rows: {len(df_left)}")
print(f"Right rows: {len(df_right)}")

# Check keys
print(f"\nLeft key duplicates: {df_left['id'].duplicated().sum()}")
print(f"Right key duplicates: {df_right['id'].duplicated().sum()}")
print(f"Left key NaN: {df_left['id'].isna().sum()}")
print(f"Right key NaN: {df_right['id'].isna().sum()}")

# Check dtypes
print(f"\nLeft key dtype: {df_left['id'].dtype}")
print(f"Right key dtype: {df_right['id'].dtype}")

# After
result = df_left.merge(df_right, on='id', how='left')
print(f"\nResult rows: {len(result)}")
print(f"Result shape: {result.shape}")
print(f"Result NaN count:\n{result.isna().sum()}")
print(f"\nResult columns: {result.columns.tolist()}")
```

<hr style="border: 2px solid black;">

# Pattern F: Shape & Dimension Errors {#pattern-f}

"You treated a Series like a DataFrame, or got confused about rows vs columns."

---

## F1: DataFrame vs Series Confusion (Single vs Double Brackets)

**Silent error: Result has unexpected dimensions.**

**Example:**
```python
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})

# Single brackets: returns a Series (1D)
series = df['a']
print(type(series))  # <class 'pandas.core.series.Series'>
print(series.shape)  # (3,)  — one dimension

# Double brackets: returns a DataFrame (2D)
df_slice = df[['a']]
print(type(df_slice))  # <class 'pandas.core.frame.DataFrame'>
print(df_slice.shape)  # (3, 1)  — rows and columns

# Multiple columns: must use double brackets
df_slice = df[['a', 'b']]
print(df_slice.shape)  # (3, 2)
```

**Why it matters:**
Some methods work on DataFrames but not Series (e.g., `.corr()` with multiple columns).

**Fix:**
```python
# If you want a DataFrame with one column
df_one_col = df[['a']]

# If you want a Series
series = df['a']
# Or convert Series back to DataFrame
df_from_series = series.to_frame(name='a')
```

---

## F2: axis=0 vs axis=1 Confusion in apply, concat, drop

**Error or wrong result: Applied function to wrong dimension.**

**Example:**
```python
df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6], 'c': [7, 8, 9]})

# axis=0: operate on rows (default)
result = df.apply(np.sum, axis=0)  # Sum each column
print(result)  # a: 6, b: 15, c: 24

# axis=1: operate on columns
result = df.apply(np.sum, axis=1)  # Sum each row
print(result)  # 0: 12, 1: 15, 2: 18

# Confusion:
df.drop('a', axis=0)  # Drop rows named 'a' (not exist)
df.drop('a', axis=1)  # Drop column 'a' (correct)
df.drop(0, axis=0)  # Drop row 0 (correct)
df.drop(0, axis=1)  # Drop column 0 (doesn't exist)
```

**Memory aid:**
- `axis=0`: rows (vertical axis)
- `axis=1`: columns (horizontal axis)

**Fix:**
```python
# Be explicit
df.sum(axis=0)  # Sum down (across rows)
df.sum(axis=1)  # Sum across (across columns)

# Or use explicit parameter names
df.drop(labels='a', axis=1)  # Drop column
df.drop(index=0)  # Drop row (explicit name)
df.drop(columns=['a'])  # Drop columns (explicit)
```

---

## F3: Boolean Indexing Shape Mismatch

**Error: Boolean mask has wrong length.**

**Example:**
```python
df = pd.DataFrame({'a': [1, 2, 3, 4, 5], 'b': [10, 20, 30, 40, 50]})

mask = df['a'] > 2  # Boolean Series with 5 elements
result = df[mask]  # Filter df by mask (OK)
print(len(result))  # 3 rows

# But if mask has different length:
mask = pd.Series([True, False, True])  # Only 3 elements
result = df[mask]  # Error! df has 5 rows, mask has 3
```

**Fix:**
```python
# Make sure mask has the same length as df
assert len(mask) == len(df), "Mask length mismatch!"

# Or create mask from the same df
mask = df['a'] > 2  # Guaranteed to have len(df) elements
result = df[mask]
```

---

## F4: concat/append with Misaligned Columns

**Silent error: Concatenation creates extra NaN columns.**

**Example:**
```python
df1 = pd.DataFrame({'a': [1, 2], 'b': [10, 20]})
df2 = pd.DataFrame({'a': [3, 4], 'c': [30, 40]})  # Has 'c' instead of 'b'

result = pd.concat([df1, df2])
print(result.shape)  # (4, 3)  — 4 rows, 3 columns (a, b, c)
print(result)
# df2 rows have NaN for 'b'; df1 rows have NaN for 'c'
```

**Fix:**
```python
# 1. Standardize columns before concat
df2 = df2.rename(columns={'c': 'b'})
result = pd.concat([df1, df2])

# 2. Or select common columns only
result = pd.concat([df1[['a']], df2[['a']]])

# 3. Check for NaN columns after concat
print(result.isna().sum())
```

<hr style="border: 2px solid black;">

# Pattern G: Silent Data Errors {#pattern-g}

"Code runs, no error, but your data is quietly wrong. These are the most dangerous."

---

## G1: dtype Coercion (int Column with NaN Becomes float64)

**Silent error: int column automatically becomes float when NaN is introduced.**

**Example:**
```python
df = pd.DataFrame({'count': [1, 2, 3, 4, 5]})
print(df['count'].dtype)  # int64

# Add a NaN somewhere
df.loc[2, 'count'] = np.nan
print(df['count'].dtype)  # float64  (changed!)
print(df)
# 1.0, 2.0, NaN, 4.0, 5.0 (all become floats)
```

**Explanation:**
Python's int can't represent NaN (only floats can). When you introduce NaN, the entire column coerces to float.

**Why it matters:**
- Integer operations (e.g., modulo, bitwise) fail on floats
- Comparisons behave differently (1.0 vs 1)
- Serialization/storage expects int

**Fix:**
```python
# 1. Detect and plan for NaN
print(df['count'].isna().any())  # True?

# 2. Use nullable integer type (Pandas 1.0+)
df['count'] = df['count'].astype('Int64')  # Note capital 'I'
print(df['count'].dtype)  # Int64 (nullable integer type)

# 3. Or fill NaN before operations
df['count'] = df['count'].fillna(-1)  # Use sentinel
df['count'] = df['count'].astype('int64')

# Verify
print(df['count'].dtype)
print(df)
```

---

## G2: Object dtype Hiding Mixed Types (Strings and Numbers in Same Column)

**Silent error: Column has mixed types (strings and ints), silently becomes object dtype.**

**Example:**
```python
df = pd.DataFrame({'value': [1, 2, 'three', 4, 5]})
print(df['value'].dtype)  # object

# Now try numeric operations
result = df['value'] + 10  # Error or weird behavior
```

**Fix:**
```python
# 1. Detect object columns
print(df.dtypes)
object_cols = df.select_dtypes(include=['object']).columns
print(object_cols)

# 2. Inspect for mixed types
for col in object_cols:
    types_in_col = df[col].apply(type).unique()
    print(f"{col}: {types_in_col}")

# 3. Clean before conversion
df['value'] = pd.to_numeric(df['value'], errors='coerce')
# Non-numeric values become NaN
print(df)
```

---

## G3: NaN Propagation in Arithmetic (NaN + 5 = NaN)

**Silent error: NaN in one cell spreads through calculations.**

**Example:**
```python
df = pd.DataFrame({'a': [1, 2, np.nan, 4], 'b': [10, 20, 30, 40]})

df['c'] = df['a'] + df['b']
print(df)
# Row 2 has NaN in 'c' (because NaN + 30 = NaN)

# Groupby with NaN
result = df.groupby('a')['b'].sum()
print(result)  # NaN group might be skipped
```

**Fix:**
```python
# 1. Detect NaN before operations
print(df['a'].isna().sum())  # How many NaN?
print(df['a'].isna().any())  # Any NaN?

# 2. Fill NaN before calculation
df['a_filled'] = df['a'].fillna(0)  # Or df['a'].fillna(df['a'].mean())
df['c'] = df['a_filled'] + df['b']

# 3. Or use .fillna() in aggregation
df_clean = df.fillna(method='ffill')  # Forward fill
result = df_clean.groupby('a')['b'].sum()

# Verify
print(df['c'].isna().sum())  # Should be 0 (or expected count)
```

---

## G4: String Method on Non-String Column (.str Accessor)

**Error or silent failure: Using .str on non-string column.**

**Example:**
```python
df = pd.DataFrame({'price': [100, 200, 300]})
print(df['price'].dtype)  # int64

# Try string operation
result = df['price'].str.upper()  # Error! int64 doesn't have .str

# But if column is object (mixed):
df2 = pd.DataFrame({'data': [1, 'two', 3]})
result = df2['data'].str.upper()  # Partially works; non-strings become NaN
```

**Fix:**
```python
# 1. Check dtype before .str access
print(df['price'].dtype)

# 2. Convert to string first
df['price_str'] = df['price'].astype(str)
result = df['price_str'].str.zfill(5)  # Now .str works

# 3. Or use apply
result = df['price'].apply(lambda x: str(x).zfill(5))
```

---

## G5: apply() Returning Unexpected Types

**Silent error: apply() infers dtype from result, leading to unexpected coercion.**

**Example:**
```python
df = pd.DataFrame({'a': [1, 2, 3, 4, 5]})

# Apply that sometimes returns int, sometimes float
def divide_by_two(x):
    if x % 2 == 0:
        return x // 2  # int
    else:
        return x / 2   # float

result = df['a'].apply(divide_by_two)
print(result.dtype)  # float64 (inferred from mix of int and float)
print(result)  # [0.5, 1.0, 1.5, 2.0, 2.5]
```

**Fix:**
```python
# 1. Ensure apply returns consistent types
def divide_by_two(x):
    return x / 2  # Always return float

result = df['a'].apply(divide_by_two)

# 2. Or specify dtype explicitly
result = df['a'].apply(divide_by_two).astype('float64')

# 3. Check result dtype after apply
print(result.dtype)
```

---

## Diagnostic Commands for Silent Errors

Run these after every transformation:
```python
# Check dimensions
print(f"Shape: {df.shape}")
print(f"Rows: {len(df)}")

# Check dtypes
print(df.dtypes)
print(df.info())

# Check for NaN
print(df.isna().sum())  # Per column
print(df.isna().sum().sum())  # Total

# Check for unexpected values
print(df.describe())
print(df.head(10))
print(df.tail(10))

# Check for duplicates
print(df.duplicated().sum())

# Sample data
print(df.sample(5))
```

<hr style="border: 2px solid black;">

# Pattern H: Import & Environment Errors {#pattern-h}

"The package isn't installed, or you imported it wrong."

(Less likely in a data wrangling interview, but worth knowing.)

---

## H1: ModuleNotFoundError (Package Not Installed)

**Error:**
```
ModuleNotFoundError: No module named 'pandas'
```

**Broken code:**
```python
import pandas  # Package not installed
```

**Fix:**
```bash
pip install pandas
```

---

## H2: ImportError (Wrong Function Name)

**Error:**
```
ImportError: cannot import name 'read_csv' from 'pandas'
```

**Broken code:**
```python
from pandas import read_csv  # Correct
from pandas import read_CSV  # Wrong capitalization
```

**Fix:**
```python
# Check function name (case-sensitive)
import pandas as pd
df = pd.read_csv('file.csv')  # Correct
```

---

## H3: Version Conflicts (Pandas API Changes)

**Error:**
```
TypeError: append() got an unexpected keyword argument
```
(In Pandas 2.0+, `.append()` was removed.)

**Broken code:**
```python
# Old Pandas (< 2.0)
df = df.append(new_row)  # Deprecated

# New Pandas (2.0+)
df = pd.concat([df, new_row])  # New way
```

**Fix:**
```python
# Check Pandas version
import pandas as pd
print(pd.__version__)

# Use compatible code
df = pd.concat([df, new_row], ignore_index=True)
```

<hr style="border: 3px solid black;">

# Section 4: Code Review Checklist {#section-4}

In an interview, you'll review code (yours or someone else's). Use this 3-pass approach:

---

## Pass 1: Structure (Imports, Naming, Readability)

| Checkpoint | Question | What to Look For |
|------------|----------|------------------|
| Imports | Are all imports at the top? | `import pandas as pd` before use |
| Naming | Are variable names descriptive? | `df_orders` not `df1`; `customer_id` not `cid` |
| Comments | Are complex steps explained? | "Merge orders with customer info to add customer names" |
| Constants | Are magic numbers extracted? | `MERGE_SUFFIX = '_left'` not hardcoded |
| Line length | Is code readable (< 100 chars)? | Long operations broken into steps |
| Functions | Is code modular (not 500 lines)? | Each function does one thing |

---

## Pass 2: Logic (Merge Types, Groupby Completeness, NaN Handling, Dtype Awareness)

| Checkpoint | Question | What to Look For |
|------------|----------|------------------|
| Merge type | Is `how=` correct? | `how='left'` to preserve all source rows; `how='inner'` only matches |
| Merge keys | Are keys checked for uniqueness? | No fan-out silently happening |
| Merge dtypes | Do join keys have the same dtype? | int vs float mismatch detected |
| Suffixes | Are overlapping columns handled? | `suffixes=('_left', '_right')` specified |
| Groupby | Does groupby handle NaN correctly? | NaN groups explicitly handled or filtered |
| NaN handling | Is NaN treatment intentional? | `.fillna()`, `.dropna()`, or `.isna().sum()` checks |
| Dtype conversions | Are columns converted to expected types? | `pd.to_numeric()`, `pd.to_datetime()` before use |
| Chained indexing | Are assignments using `.loc[]`? | No `df[mask]['col'] = value` |

---

## Pass 3: Data Correctness (Row Counts, Edge Cases, Chained Operations)

| Checkpoint | Question | What to Look For |
|------------|----------|------------------|
| Row counts | Before/after row counts reasonable? | `assert len(result) <= len(df_left)` for inner merges |
| Empty data | What if df is empty? | Handle `len(df) == 0` gracefully |
| NaN edge cases | What if all rows are NaN? | Explicit check and plan |
| Duplicate keys | Are duplicates in join keys handled? | `.drop_duplicates()` before merge or aggregate |
| Column order | Does order matter in output? | Reorder columns explicitly if needed |
| Final shape | Is final df shape expected? | `.shape`, `.columns`, `.dtypes` match spec |
| Chained ops | Can each step be tested independently? | Break long chains into variables |
| Sample test | Does code work on a small sample? | Always test on head() first |

---

## Interview Tip: Narrate Your Review

Don't just read silently. Say what you're checking:

✅ Good: "I see imports at the top. Good. Now let me check the merge... The keys are 'customer_id', let me verify they have the same dtype... Left key is int64, right key is float64. That's a problem. We need to convert before merging."

❌ Bad: Stare at code for 20 seconds, then say "Looks good."

Narration shows you know what to look for and builds interviewer confidence.

<hr style="border: 3px solid black;">

# Section 5: Debugging Workflow Under Pressure {#section-5}

You have 60 minutes. Code breaks. Here's the fastest path to a fix.

---

## The 5-Step Workflow

<div class="fc">
  <div class="fc-node fc-start">Code Breaks!</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action"><strong>STEP 1: READ</strong><br/>Read the error message bottom-up. 3-second read.</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action"><strong>STEP 2: LOCATE</strong><br/>Go to the exact line in the traceback.</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action"><strong>STEP 3: ISOLATE</strong><br/>Inspect with print(), .shape, .head(), .dtypes<br/>What does the data look like right before the break?</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-action"><strong>STEP 4: FIX</strong><br/>Fix the root cause, not the symptom.<br/>Apply fix pattern from the library above.</div>
  <div class="fc-arrow">▼</div>
  <div class="fc-node fc-end"><strong>STEP 5: VERIFY</strong><br/>Assertions, row counts, dtypes.<br/>Code runs, data correct.</div>
</div>

---

## Isolation Techniques (STEP 3)

When code breaks, inspect:

| Technique | Command | Why It Helps |
|-----------|---------|-------------|
| Shape | `print(df.shape)` | How many rows/cols? Is data empty? |
| First rows | `print(df.head(3))` | What does the data actually look like? |
| Dtypes | `print(df.dtypes)` | Are columns the right types? |
| Info | `df.info()` | Memory, NaN count, dtype summary |
| Describe | `df.describe()` | Min/max/mean (catch outliers, bad ranges) |
| NaN count | `df.isna().sum()` | Which columns have missing data? |
| Columns | `print(df.columns)` | What columns exist? Any typos? |
| Index | `print(df.index)` | Is the index what you expect? |
| Unique vals | `df['col'].unique()` | How many distinct values? |
| Value counts | `df['col'].value_counts()` | Distribution of categories |
| Sample | `df.sample(5)` | Random rows (shows different data than head) |

---

## What to Say Out Loud (During Interview)

**When error happens:**

✅ "I see a KeyError on line 15. The error message says 'naem'. Looking at line 15, I'm trying to access df['naem'], but that's a typo. Let me check what columns exist..."

❌ "Hmm, KeyError... [long pause]"

**When investigating:**

✅ "Let me check the shape before and after the merge. Left df has 1000 rows, right df has 500 rows. After merge with how='left', I get 1000 rows. That's good—no rows were dropped. But let me verify the join key matches by checking the unique values in each..."

❌ "Let me print some stuff..." [types df.head() without explanation]

---

## Verification (STEP 5)

After fixing, assert your expectations:

```python
# Row counts
assert len(result) == len(df_left), "Lost rows in merge!"
assert result.shape[1] == 5, "Wrong column count!"

# Dtypes
assert result['amount'].dtype in ['int64', 'float64'], "Unexpected dtype!"

# NaN
assert result[['id', 'name']].isna().sum().sum() == 0, "Key columns have NaN!"

# Range
assert result['age'].min() >= 0, "Age is negative!"
assert result['age'].max() <= 120, "Age is unrealistic!"

# Uniqueness (if needed)
assert not result['customer_id'].duplicated().any(), "Duplicate customers!"

print("All assertions passed! ✓")
```

**Why assertions matter in interviews:**
- Shows you think about edge cases
- Prevents silent bugs from going unnoticed
- Makes you look methodical

<hr style="border: 3px solid black;">

# Section 6: Common Interview Traps {#section-6}

The interviewer will intentionally plant bugs or edge cases. Here are the top traps:

| Trap | What the Interviewer Planted | What They Want to Hear |
|------|------------------------------|------------------------|
| **Chained indexing** | `df[df['x'] > 5]['y'] = 10` produces SettingWithCopyWarning | "This creates a view, not a copy. We should use `.loc[]` instead." |
| **Merge fan-out** | Left and right both have duplicate keys. Result has way more rows. | "I need to check key uniqueness before merging. Let me aggregate duplicates or use `.drop_duplicates()` first." |
| **NaN in groupby** | `df.groupby('category')` — but some category values are NaN. They silently get dropped. | "NaN values are skipped by groupby. If we need them, use `.groupby('category', dropna=False)` or handle explicitly." |
| **Integer column with NaN** | Assign NaN to an int column. It silently becomes float64. | "Int columns can't have NaN. Either use nullable Int64, or handle NaN with `.fillna()` and keep as int." |
| **inplace=True return value** | `df = df.drop('column', inplace=True)` assigns None, not the df. | "`inplace=True` returns None. Either use `inplace=True` without assignment, or assign the result of the regular operation." |
| **Missing .copy()** | Modify a filtered slice: `subset = df[df['x'] > 5]; subset['y'] = 10` | "This modifies a view. We need `.copy()` or use `.loc[]`." |
| **Wrong merge type** | Code uses `how='inner'`, losing unmatched rows. But the goal was `how='left'`. | "Inner merge only keeps matching rows. For a left join preserving all source rows, use `how='left'`." |
| **axis confusion** | `df.drop(0, axis=0)` when they meant to drop column 0. | "axis=0 is rows, axis=1 is columns. For clarity, use `df.drop(columns=['col'])` or `df.drop(index=0)`." |
| **Dtype mismatch in merge** | Left key is int64, right key is float64. Merge returns 0 rows. | "Join keys must have the same dtype. Convert: `df['id'] = df['id'].astype('int64')` before merging." |
| **Silent dtype coercion** | Apply a numeric function to a string column. No error, just weird results. | "Always check `.dtypes` and use `pd.to_numeric()` or `.astype()` before math operations." |
| **Overlapping column names** | Merge without suffixes specified. Result has `_x` and `_y` suffixes everywhere. | "Specify `suffixes=('_left', '_right')` or rename columns before merging for clarity." |
| **Pandas version differences** | Code uses `.append()`. In Pandas 2.0+, it was removed. | "Check Pandas version and use `pd.concat()` instead of deprecated `.append()`." |

---

## Strategy: When You Spot a Trap

1. **Point it out immediately:** "I notice this line is using chained indexing, which might produce a warning."
2. **Explain why it's wrong:** "Chained indexing creates a view, not a copy, so the assignment is ambiguous."
3. **Propose the fix:** "We should use `.loc[]` instead: `df.loc[df['x'] > 5, 'y'] = 10`."
4. **Show you'd verify:** "And I'd test it to make sure the assignment actually persists."

This turns a potential mistake into a sign you're careful and knowledgeable.